# Budgerigar S0：连续听觉前端与 4 层 token 记忆
底层滤波跨任意音频 chunk 连续，4 层递归 token 每 10 ms 优化；听完单个数字后在允许窗口内自主输出 token。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip();print('commit:',commit)

In [ ]:
#@title 2. Drive、manifest 与双记忆连续流接口检查
from google.colab import drive
drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar');MANIFEST=WORK_ROOT/'manifests'/'fsdd.jsonl'
import torch,json
if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime')
from budgerigar.streaming_short_memory import ShortMemoryConfig,create_short_memory_model,WaveformTickBuffer
config=ShortMemoryConfig(sample_rate=16000,tick_samples=160,hidden_dim=96,token_layers=4,attention_heads=4,acoustic_slots=8,acoustic_reconstruction_slots=8,acoustic_bins=32);model=create_short_memory_model(config)
dummy=torch.randn(2,7,160)
with torch.no_grad():batched,state,_=model(dummy);stream_state=None;parts=[]
with torch.no_grad():
 for tick in range(dummy.shape[1]):value,stream_state,_=model.stream_step(dummy[:,tick],stream_state);parts.append(value)
difference=float((batched-torch.stack(parts,1)).abs().max());print('parameters:',sum(p.numel() for p in model.parameters()),'difference:',difference);assert difference<1e-5
buffer=WaveformTickBuffer(160);assert len(buffer.push(torch.zeros(77)))==0 and len(buffer.push(torch.zeros(243)))==2

In [ ]:
#@title 3. T4 dual semantic-acoustic memory training
MAX_STEPS=500 #@param {type:'integer'}
BATCH_SIZE=16 #@param {type:'integer'}
from budgerigar.train_short_memory import ShortMemoryTrainingConfig,train_short_memory
BASE_RUN_DIR=WORK_ROOT/'checkpoints'/'streaming_hierarchical_memory_decoder_s0'
RUN_DIR=WORK_ROOT/'checkpoints'/'streaming_dual_semantic_acoustic_memory_s0';training=ShortMemoryTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,max_train_records=1000,max_validation_records=300,reconstruction_weight=1.0,acoustic_contrastive_weight=1.0)
report=train_short_memory(MANIFEST,RUN_DIR,training,config,initialize_from=BASE_RUN_DIR/'best.pt');print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 双记忆初步门槛
best=max(report['history'],key=lambda x:x['validation_end_digit_accuracy']+x['validation_memory_digit_accuracy']+x['validation_reconstruction_mean_l1']-x['validation_reconstruction_l1'])
s0_pass=best['validation_end_digit_accuracy']>.8 and best['validation_memory_digit_accuracy']>.8 and best['validation_reconstruction_l1']<best['validation_reconstruction_shuffled_l1'] and best['validation_reconstruction_l1']<best['validation_reconstruction_mean_l1'] and best['validation_early_emission_rate']<.01 and best['streaming_max_logit_difference']<1e-4
print(json.dumps(best,ensure_ascii=False,indent=2));print('s0_pass =',s0_pass)
from budgerigar.experiment import write_run_metadata
write_run_metadata(RUN_DIR/'run_metadata.json',MANIFEST,{'architecture':'streaming_cochlear_dual_semantic_acoustic_memory','semantic_layers':4,'acoustic_slots':8,'acoustic_reconstruction_slots':8,'high_level_tick_ms':10,'s0_pass':s0_pass,**best},repository=REPO_DIR)
if not s0_pass:print('未通过：保持语义主干，按声学正确目标与平均模板差距继续修正。')

In [ ]:
#@title 5. 分别试听完整输入和模型输出时间轴
EXAMPLE_INDEX=0 #@param {type:'integer'}
from budgerigar.audition_short_memory import render_short_memory_audition
from IPython.display import Audio,display
audit=render_short_memory_audition(MANIFEST,RUN_DIR/'best.pt',RUN_DIR/'auditions',EXAMPLE_INDEX)
print(json.dumps(audit,ensure_ascii=False,indent=2))
print('输入（包含完整静默）：');display(Audio(filename=audit['input_path']))
print('模型输出（同长度、同时间轴）：');display(Audio(filename=audit['output_path']))
print('注意：输出是模型实际 token 的音高映射，不是语音合成；无发射时输出保持静默。')

In [ ]:
#@title 6. 内容归纳、短期保持与解码审计
from budgerigar.audit_short_memory import audit_short_memory
AUDIT_PATH=RUN_DIR/'memory_audit.json'
memory_audit=audit_short_memory(MANIFEST,RUN_DIR/'best.pt',AUDIT_PATH,batch_size=16,max_records=300)
print(json.dumps(memory_audit,ensure_ascii=False,indent=2));print('report:',AUDIT_PATH)
print('semantic_memory_pass =',memory_audit['semantic_memory_pass']);print('acoustic_memory_pass =',memory_audit['acoustic_memory_pass'])
if not memory_audit['audit_pass']:print('联合审计未通过：不要进入波形复读，先按具体消融失败项修正。')